# Pyxa → SpatialData demo

Reads the `xsmall` crop (a 100 × 100 × 100 µm cube, 187 cells) of the
[Stellaromics/demo](https://huggingface.co/datasets/Stellaromics/demo) dataset with the experimental
`pyxa` reader from [spatialdata-io](https://github.com/scverse/spatialdata-io), writes it to SpatialData
Zarr, and browses it one z-plane at a time.

In [ ]:
import zipfile
from pathlib import Path

import fsspec
import ipywidgets as widgets
import matplotlib.pyplot as plt
from spatialdata import get_extent
from spatialdata.transformations import get_transformation

from spatialdata_io.experimental import pyxa

## 1. Get the data

Downloads `xsmall/` from the Hugging Face Hub through fsspec's `hf://` filesystem (provided by
`huggingface_hub`), then extracts the zipped OME-Zarr DAPI mosaic next to the other files.

In [ ]:
DATA_DIR = Path("data/xsmall")
DATA_DIR.mkdir(parents=True, exist_ok=True)

fs = fsspec.filesystem("hf")
for remote_path in fs.ls("datasets/Stellaromics/demo/xsmall", detail=False):
    fs.get(remote_path, str(DATA_DIR / Path(remote_path).name))

with zipfile.ZipFile(DATA_DIR / "mosaic_3d.ome.zarr.zip") as zf:
    zf.extractall(DATA_DIR)
image_path = DATA_DIR / "mosaic_3d.ome.zarr"

sorted(p.name for p in DATA_DIR.iterdir())

## 2. Convert to SpatialData

The reader returns every element in µm in the `global` coordinate system. Segmentation polygons are
stored on disk in pixels, one per cell per z-plane; the reader converts and repairs them into
`cell_boundaries_z` (per plane, with the plane centre in a `Z_um` column) and `cell_boundaries` (one
footprint per cell, annotated by the `rna` table). The result is written to SpatialData Zarr.

In [ ]:
sdata = pyxa(DATA_DIR, image_path=image_path)
sdata.write("data/xsmall.zarr", overwrite=True)
sdata

## 3. Browse z-planes

Each slider step is one DAPI z-plane. Polygons from the same plane and transcripts within half a plane of it
are drawn on top, in blue if assigned to a cell and orange if not.

In [ ]:
# the mosaic is multiscale; browse the full-resolution level
image = sdata["mosaic_image"]["scale0"]["image"]
shapes = sdata["cell_boundaries_z"]
transcripts = sdata["transcripts"].compute()
extent = get_extent(image)

# image plane k covers [z0 + k * dz, z0 + (k + 1) * dz) µm
affine = get_transformation(image, to_coordinate_system="global").to_affine_matrix(
    input_axes=("z", "y", "x"), output_axes=("z", "y", "x")
)
z0, dz = affine[0, 3], affine[0, 0]


def show_plane(k: int) -> None:
    z = z0 + (k + 0.5) * dz
    polygons = shapes[(shapes["Z_um"] - z).abs() <= dz / 2]
    near = transcripts[(transcripts["z"] - z).abs() <= dz / 2]
    assigned = near["assigned"]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(
        image.isel(c=0, z=k).values,
        cmap="gray",
        extent=(extent["x"][0], extent["x"][1], extent["y"][1], extent["y"][0]),
    )
    if len(polygons):
        polygons.boundary.plot(ax=ax, color="white", linewidth=0.6)
    ax.scatter(near.loc[assigned, "x"], near.loc[assigned, "y"], s=8, c="#2a78d6", label="assigned")
    ax.scatter(near.loc[~assigned, "x"], near.loc[~assigned, "y"], s=8, c="#eb6834", label="unassigned")
    ax.set(
        xlim=extent["x"],
        ylim=(extent["y"][1], extent["y"][0]),
        xlabel="x (µm)",
        ylabel="y (µm)",
        title=f"z = {z:.2f} µm (plane {k}): {len(polygons)} polygons, {len(near)} transcripts",
    )
    ax.legend(loc="upper right")
    plt.show()


n_planes = image.sizes["z"]
_ = widgets.interact(
    show_plane,
    k=widgets.IntSlider(value=n_planes // 2, min=0, max=n_planes - 1, description="z-plane", continuous_update=False),
)